# AURORA ? Colab Max

Heavy Colab runner for the AURORA Omega valuation foundation model.

This notebook is a control plane: it clones the repo, stores raw data/artifacts in Google Drive, builds a larger FMP point-in-time panel, trains Omega on GPU, evaluates against deterministic lenses, and exports Valuation MRI artifacts.


## Runtime request

Use the strongest runtime available.

- Best: A100 GPU + High RAM.
- Good: L4 GPU + High RAM.
- Smoke: T4 GPU with fewer tickers/epochs.
- Drive storage: reserve 5-25 GB depending on ticker count.

If Colab disconnects, rerun from the top. FMP raw JSON and panel parquet are cached in Drive.


In [ ]:
import os, sys, json, time, random, subprocess, importlib.util
from pathlib import Path
from getpass import getpass

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Minimal installs. Colab usually already has torch/sklearn/pandas.
def ensure(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])
for m,p in [('yfinance','yfinance'),('pyarrow','pyarrow'),('tqdm','tqdm'),('requests','requests')]:
    ensure(m,p)

import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error

print('Torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Configuration. Push repo changes before running if Colab cannot see aurora_omega.
REPO_URL = 'https://github.com/tbasaure-sys/fin.git'
REPO_REF = 'main'  # change to a branch/commit if needed
WORKDIR = Path('/content/fin') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/blsprime_aurora_omega') if IN_COLAB else Path('./_local_data/blsprime_aurora_omega')

TARGET_TICKERS = 1500       # A100/L4: 1000-3000; smoke: 250-500
START_YEAR = 2005
LAST_FEATURE_YEAR = 2024
FORCE_PANEL_REBUILD = False
REQUEST_PAUSE_SECONDS = 0.04

EPOCHS = 120 if torch.cuda.is_available() else 30
BATCH_SIZE = 512 if torch.cuda.is_available() else 256
D_MODEL = 192 if torch.cuda.is_available() else 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 7

for p in [DRIVE_ROOT, DRIVE_ROOT/'router_cache', DRIVE_ROOT/'artifacts']:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Drive root:', DRIVE_ROOT)
print('Training:', {'epochs':EPOCHS, 'batch':BATCH_SIZE, 'd_model':D_MODEL, 'device':DEVICE})


In [ ]:
# Clone/pull repo.
if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(WORKDIR)])
    subprocess.check_call(['git', '-C', str(WORKDIR), 'fetch', '--all'])
    subprocess.check_call(['git', '-C', str(WORKDIR), 'checkout', REPO_REF])
    subprocess.call(['git', '-C', str(WORKDIR), 'pull', '--ff-only'])

sys.path.insert(0, str(WORKDIR))
missing = [p for p in [WORKDIR/'aurora_omega', WORKDIR/'scripts'/'run_aurora_router_local.py'] if not p.exists()]
if missing:
    raise RuntimeError(f'Missing repo files in Colab: {missing}. Push current AURORA Omega files or change REPO_REF.')
print('Repo ready:', WORKDIR)


In [ ]:
# Keys. Kept in memory only; not written to artifacts.
FMP_API_KEY = os.environ.get('FMP_API_KEY') or getpass('FMP API key: ')
os.environ['FMP_API_KEY'] = FMP_API_KEY
print('FMP configured:', bool(FMP_API_KEY))


## Universe from FMP

The notebook tries FMP stock screener first. If the plan blocks that endpoint, it falls back to the repo core universe.


In [ ]:
FMP_STABLE = 'https://financialmodelingprep.com/stable'

def fmp_get(endpoint, params=None, retries=4):
    params = dict(params or {})
    params['apikey'] = FMP_API_KEY
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(f'{FMP_STABLE}/{endpoint}', params=params, timeout=45)
            if r.status_code == 429:
                time.sleep(3 * (attempt + 1)); continue
            r.raise_for_status(); return r.json()
        except Exception as e:
            last = e; time.sleep(1.5 * (attempt + 1))
    raise last

def load_universe(target=TARGET_TICKERS):
    cache = DRIVE_ROOT / f'universe_{target}.json'
    if cache.exists() and not FORCE_PANEL_REBUILD:
        return json.loads(cache.read_text())
    try:
        payload = fmp_get('stock-screener', {'exchange':'NASDAQ,NYSE,AMEX','isActivelyTrading':'true','marketCapMoreThan':500000000,'limit':max(target*2,1000)})
        rows = payload if isinstance(payload, list) else payload.get('data', []) if isinstance(payload, dict) else []
        rows = [r for r in rows if isinstance(r, dict) and r.get('symbol')]
        rows = sorted(rows, key=lambda r: float(r.get('marketCap') or 0), reverse=True)
        tickers = [str(r['symbol']).upper() for r in rows]
    except Exception as e:
        print('FMP screener failed, using core universe:', repr(e))
        import scripts.run_aurora_router_local as router
        tickers = sorted(set(router.CORE_UNIVERSE))
    cleaned=[]
    for t in tickers:
        if t and '^' not in t and '/' not in t and t not in cleaned:
            cleaned.append(t)
    cleaned = cleaned[:target]
    cache.write_text(json.dumps(cleaned, indent=2))
    return cleaned

TICKERS = load_universe()
print('Ticker count:', len(TICKERS), TICKERS[:15])


## Build point-in-time panel using repo pipeline

This patches repo paths/constants so raw FMP JSON and artifacts live in Drive instead of Colab temporary storage.


In [ ]:
import scripts.run_aurora_router_local as router

# Patch global paths/constants to Drive and wider history.
router.LOCAL_ROOT = DRIVE_ROOT / 'router_cache'
router.RAW_ROOT = router.LOCAL_ROOT / 'raw_fmp'
router.ARTIFACT_ROOT = DRIVE_ROOT / 'router_artifacts'
router.START_YEAR = START_YEAR
router.LAST_FEATURE_YEAR = LAST_FEATURE_YEAR
router.PANEL_VERSION = f'omega_colab_max_{START_YEAR}_{LAST_FEATURE_YEAR}'
os.environ['FMP_REQUEST_PAUSE_SECONDS'] = str(REQUEST_PAUSE_SECONDS)

panel = router.build_or_load_panel(FMP_API_KEY, TICKERS, force=FORCE_PANEL_REBUILD)
featured = router.add_lens_predictions(router.add_features(panel))
featured['omega_regime'] = featured.apply(router.classify_spine_regime, axis=1)
featured['omega_primary_question'] = featured['omega_regime'].map(router.primary_question_for_regime)
exps = featured.apply(router.reverse_dcf_expectations, axis=1)
featured['omega_expectations_pressure'] = [e['valuation_pressure_score'] for e in exps]
featured['omega_feasibility_score'] = [router.score_expectation_feasibility(row, e)['score'] for (_, row), e in zip(featured.iterrows(), exps)]
featured['omega_downside_anchor_score'] = [router.anchor_lens_checks(row, router.classify_spine_regime(row), e)['asset_value']['score'] for (_, row), e in zip(featured.iterrows(), exps)]

panel_path = DRIVE_ROOT / f'featured_panel_{router.PANEL_VERSION}_{len(TICKERS)}.parquet'
featured.to_parquet(panel_path, index=False)
print('Featured panel:', featured.shape, 'tickers:', featured.ticker.nunique())
print('Saved:', panel_path)
featured[['ticker','year','omega_regime','pred_reverseDcf','pred_assetValue','ann_return_3y_fwd']].tail()


## Train AURORA Omega

This uses the repo `aurora_omega` package. Omega remains shadow-only unless it beats deterministic spine and single-lens baselines.


In [ ]:
from aurora_omega.data import build_omega_bundle, LENS_NAMES
from aurora_omega.train import TrainConfig, train_omega, evaluate_omega
from aurora_omega.outputs import write_valuation_mri

stamp = pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')
out_dir = DRIVE_ROOT / 'artifacts' / stamp
out_dir.mkdir(parents=True, exist_ok=True)

bundle = build_omega_bundle(featured, max_years=10, train_end_year=router.TRAIN_END_YEAR, val_start_year=router.VAL_START_YEAR)
cfg = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=8e-4, d_model=D_MODEL, device=DEVICE, seed=SEED)
model, train_report = train_omega(bundle, cfg, out_dir)
eval_out = evaluate_omega(model, bundle, cfg)
memos = write_valuation_mri(bundle, eval_out, out_dir)
print('Validation:', eval_out['metrics'])
print('MRI count:', len(memos), 'out_dir:', out_dir)


## Baseline checks

Compare Omega MoE against simple deterministic lenses. Do not promote if it does not beat them.


In [ ]:
def spearman_by_year(frame, pred_col, target_col):
    vals=[]
    for _, sub in frame.groupby('year'):
        if len(sub) >= 8 and sub[pred_col].nunique() > 1 and sub[target_col].nunique() > 1:
            c = sub[[pred_col,target_col]].corr(method='spearman').iloc[0,1]
            if np.isfinite(c): vals.append(float(c))
    return float(np.mean(vals)) if vals else np.nan

val = bundle.frame.iloc[eval_out['row_index']].copy().reset_index(drop=True)
val['omega_moe_3y'] = eval_out['omega_return']
val['omega_pred_3y'] = eval_out['pred_returns'][:,1]
checks = {}
for col in ['pred_assetValue','pred_reverseDcf','pred_residualIncome','pred_capitalCycle','omega_moe_3y','omega_pred_3y']:
    sub = val.dropna(subset=[col,'ann_return_3y_fwd'])
    checks[col] = {'mae_3y':float(mean_absolute_error(sub['ann_return_3y_fwd'], sub[col])), 'ic_3y':spearman_by_year(sub, col, 'ann_return_3y_fwd')}
checks_df = pd.DataFrame(checks).T.sort_values('mae_3y')
checks_df


In [ ]:
manifest = {
    'mode':'aurora_omega_colab_max_control_plane',
    'created_at':pd.Timestamp.utcnow().isoformat(),
    'artifact_dir':str(out_dir),
    'drive_root':str(DRIVE_ROOT),
    'repo_ref':REPO_REF,
    'target_tickers':TARGET_TICKERS,
    'actual_tickers':int(featured.ticker.nunique()),
    'panel_rows':int(len(featured)),
    'train_rows':len(bundle.train),
    'val_rows':len(bundle.val),
    'device':DEVICE,
    'epochs':EPOCHS,
    'batch_size':BATCH_SIZE,
    'd_model':D_MODEL,
    'omega_metrics':eval_out['metrics'],
    'baseline_checks':checks,
    'mri_count':len(memos),
    'production_candidate':False,
    'status':'shadow_only_until_gates_pass',
}
(out_dir/'colab_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
(DRIVE_ROOT/'LATEST_OMEGA_ARTIFACT.txt').write_text(str(out_dir))
print(json.dumps(manifest, indent=2, default=str)[:5000])
print('Saved latest pointer:', DRIVE_ROOT/'LATEST_OMEGA_ARTIFACT.txt')


## Gates

Keep Omega shadow unless:

- `omega_moe_3y` beats `assetValue`, `reverseDcf`, and `spine_v1` by a real margin.
- IC is positive by year and stable.
- Decile spread is positive in every validation year.
- The router does not collapse into one lens.
- Valuation MRI abstention is meaningful.

If it fails, the next unlock is not just more epochs: expand the panel and add stronger pretraining/text/graph data.
